# Memory Architecture — Episodic, Semantic, and Procedural Memory Tiers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/88-memory-architecture/memory_architecture_workbook.ipynb)

Inspired by cognitive science: humans use multiple memory systems simultaneously. This workshop maps three biological memory types to LLM agent patterns, building an agent that stores facts, events, and behavioural instructions in separate tiers.

---

### Workshop Roadmap

| # | Section |
|---|----------|
| 1 | **Three Tiers** — episodic, semantic, procedural |
| 2 | **MemoryStore** — TypedDict structure |
| 3 | **Classifier Node** — LLM-based tier routing |
| 4 | **Respond Node** — using all three tiers |
| 5 | **Production Architecture** — external storage backends |
| ★ | **Exercises** — working memory tier, forget function |

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'langgraph', 'langchain-openai', 'python-dotenv'])
    print('Colab install complete.')
else:
    print('Local environment — skipping install.')

In [ ]:
import os

# Colab stores secrets in userdata; local dev uses a .env file
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get('OPENAI_API_KEY', '')
if not key:
    raise RuntimeError('OPENAI_API_KEY is required for the live memory-tier cells.')
print('API key loaded.')

## Part 1 — Three Memory Tiers

Cognitive science identifies three distinct memory systems. Each maps naturally to a different agent memory pattern:

| Tier | Cognitive parallel | What it stores | Retrieval strategy |
|------|---------------------|----------------|--------------------|
| **Episodic** | "What happened" | Events, conversation turns | Recency (last N) |
| **Semantic** | "What I know" | Facts, preferences, attributes | Key lookup |
| **Procedural** | "How to do it" | Behavioural patterns, instructions | All patterns |

**Examples:**
- Episodic: `"User asked about Python at 3pm"`
- Semantic: `{"user_name": "Sam", "favorite_language": "Rust"}`
- Procedural: `"Always respond with bullet points"`

A flat Redis list (example 82) mixes all three. Separating them lets the agent use each tier differently: semantic facts need key lookup, procedural rules are always active, episodic events need recency ordering.

In [ ]:
from typing import TypedDict

class MemoryStore(TypedDict):
    '''Three memory tiers stored in LangGraph state.

    In production: episodic → Redis list, semantic → key-value store, procedural → config DB.
    Here: all in-process state to focus on architecture, not plumbing.
    '''
    episodic:   list[str]       # ordered event log
    semantic:   dict[str, str]  # extracted facts
    procedural: list[str]       # behavioural patterns

# Start with empty memory
memory: MemoryStore = {
    'episodic':   [],
    'semantic':   {},
    'procedural': [],
}
print('Empty memory initialized:')
print(f'  episodic:   {memory["episodic"]}')
print(f'  semantic:   {memory["semantic"]}')
print(f'  procedural: {memory["procedural"]}')

## Part 2 — The Classifier Node

Instead of hardcoded routing rules (`if "my name is" → semantic`), we use an **LLM classifier**:

**Benefits:**
- Handles ambiguous inputs (`"I prefer bullet points"` — is that procedural or semantic?)
- Extensible — adding a new tier only requires updating the prompt
- Handles natural language variation without regex maintenance

The classifier reads the input and returns exactly one tier name. Two additional prompts extract the structured data: `key=value` for semantic facts, and a short instruction for procedural patterns.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0)

CLASSIFIER_PROMPT = '''Classify this information into ONE memory tier.

Information: {information}

Rules:
- episodic:   raw events or things that happened ("I went to the store", "My meeting ended")
- semantic:   facts about the user or world ("My name is X", "I prefer Y")
- procedural: behavioural instructions ("Always respond in French", "Use bullet points")

Reply with exactly one word: episodic, semantic, or procedural.'''

SEMANTIC_EXTRACT_PROMPT = '''Extract a key=value fact from this text.
Text: {text}
Reply as: key=value (example: user_name=Alice or favorite_language=Python).
If no clear fact, reply: none=none'''

PROCEDURAL_EXTRACT_PROMPT = '''Extract a behavioural instruction from this text.
Text: {text}
Reply with a short actionable instruction (example: Always greet user by name).
If no clear instruction, reply: none'''

def classify_and_store(text: str, memory: MemoryStore) -> tuple[MemoryStore, str]:
    '''Classify text and store in the appropriate tier. Returns updated memory + tier name.'''
    tier = llm.invoke([HumanMessage(content=CLASSIFIER_PROMPT.format(information=text))]).content.strip().lower()

    updated = dict(memory)  # shallow copy — we'll update specific keys

    if 'semantic' in tier:
        kv_raw = llm.invoke([HumanMessage(content=SEMANTIC_EXTRACT_PROMPT.format(text=text))]).content.strip()
        if '=' in kv_raw and not kv_raw.startswith('none'):
            key, _, value = kv_raw.partition('=')
            updated['semantic'] = {**updated['semantic'], key.strip(): value.strip()}
    elif 'procedural' in tier:
        pattern = llm.invoke([HumanMessage(content=PROCEDURAL_EXTRACT_PROMPT.format(text=text))]).content.strip()
        if pattern.lower() != 'none':
            updated['procedural'] = [*updated['procedural'], pattern]
    else:
        updated['episodic'] = [*updated['episodic'], text]

    return updated, tier.split()[0]

# Test the classifier on a few inputs
test_inputs = [
    'My name is Alex and I\'m a software engineer.',
    'Please always respond with bullet points.',
    'I just finished reading the LangGraph docs.',
]
for inp in test_inputs:
    updated_mem, tier = classify_and_store(inp, memory)
    memory = updated_mem
    print(f'Input: \'{inp[:50]}...\' → tier: {tier}')

print(f'\nMemory after classification:')
print(f'  episodic:   {memory["episodic"]}')
print(f'  semantic:   {memory["semantic"]}')
print(f'  procedural: {memory["procedural"]}')

## Part 3 — The Respond Node

When the agent responds, it uses all three tiers **differently**:

- **Episodic** — last 5 events for recent context (recency window)
- **Semantic** — all known facts as `key=value` pairs (precise, always current)
- **Procedural** — all active patterns (always applied — they're standing instructions)

Separating tiers means: a new procedural instruction (`"respond in French"`) doesn't push old episodic events out of a fixed-size window. Each tier manages its own size independently.

In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class AgentState(TypedDict):
    input: str
    memory: MemoryStore
    answer: str

def classify_memory_node(state: AgentState) -> dict:
    updated_mem, tier = classify_and_store(state['input'], state['memory'])
    print(f'  [classify] → {tier}')
    return {'memory': updated_mem}

def respond_node(state: AgentState) -> dict:
    m = state['memory']
    parts = []

    # Episodic: most recent 5 events
    if m['episodic']:
        parts.append('Recent events:\n' + '\n'.join(f'- {e}' for e in m['episodic'][-5:]))

    # Semantic: all known facts
    if m['semantic']:
        facts = ', '.join(f'{k}={v}' for k, v in m['semantic'].items())
        parts.append(f'Known facts about user: {facts}')

    # Procedural: all active instructions (always applied)
    if m['procedural']:
        parts.append('Active instructions:\n' + '\n'.join(f'- {p}' for p in m['procedural']))

    context = '\n\n'.join(parts) if parts else '(no memory yet)'
    prompt = f'Memory:\n{context}\n\nUser: {state["input"]}\nRespond helpfully.'
    response = llm.invoke([HumanMessage(content=prompt)])
    return {'answer': response.content.strip()}

builder = StateGraph(AgentState)
builder.add_node('classify_memory', classify_memory_node)
builder.add_node('respond',         respond_node)
builder.add_edge(START,             'classify_memory')
builder.add_edge('classify_memory', 'respond')
builder.add_edge('respond',         END)
app = builder.compile()

print('Three-tier memory graph compiled.')
print('Flow: START → classify_memory → respond → END')

## Part 4 — Running the Agent

Each turn:
1. The input is classified and stored in the appropriate tier
2. The agent responds using all three tiers as context
3. `result["memory"]` is passed forward as the next turn's input

The "What do you know about me?" query demonstrates all three tiers working together: the agent recalls facts (semantic), past events (episodic), and formatting instructions (procedural).

In [ ]:
# Reset to empty memory for clean demo
memory = {'episodic': [], 'semantic': {}, 'procedural': []}

INPUTS = [
    'My name is Sam and I\'m a data engineer.',
    'Please always respond with bullet points.',
    'I had a great team meeting this morning.',
    'My favourite database is PostgreSQL.',
    'What do you know about me so far?',
]

for user_input in INPUTS:
    print(f'\nUser: {user_input}')
    result = app.invoke({'input': user_input, 'memory': memory, 'answer': ''})
    memory = result['memory']   # carry memory forward
    print(f'Agent: {result["answer"][:200]}')

print('\n=== Final Memory State ===')
print(f'Episodic  ({len(memory["episodic"])} events):      {memory["episodic"]}')
print(f'Semantic  ({len(memory["semantic"])} facts):       {memory["semantic"]}')
print(f'Procedural ({len(memory["procedural"])} patterns): {memory["procedural"]}')

## Part 5 — Production Architecture

In production, each tier maps to a different storage backend optimised for its access pattern:

| Tier | Backend | Why |
|------|---------|-----|
| **Episodic** | Redis list | Append-only, TTL, O(1) push, easy summarisation |
| **Semantic** | PostgreSQL / Redis hash | Key lookup, updatable, structured queries |
| **Procedural** | Config DB / file | Persistent, user-controlled, rarely changes |

The LangGraph `AgentState.memory` is a **runtime buffer** — loaded from these stores at session start, written back at session end. The `classify → store → respond` pattern is identical; only the persistence layer changes.

In [ ]:
# Production pattern: load all three tiers from external stores at session start
# Here we simulate it with dicts/lists

# Simulated external stores
EPISODIC_STORE:   dict[str, list[str]] = {'user_sam': ['Had team meeting', 'Reviewed PR']}
SEMANTIC_STORE:   dict[str, dict]      = {'user_sam': {'user_name': 'Sam', 'role': 'data_engineer'}}
PROCEDURAL_STORE: dict[str, list[str]] = {'user_sam': ['Always use bullet points', 'Be concise']}

def load_all_memory(user_id: str) -> MemoryStore:
    '''Load all three tiers from external stores into LangGraph state.'''
    return {
        'episodic':   EPISODIC_STORE.get(user_id, []),
        'semantic':   SEMANTIC_STORE.get(user_id, {}),
        'procedural': PROCEDURAL_STORE.get(user_id, []),
    }

def save_all_memory(user_id: str, memory: MemoryStore) -> None:
    '''Persist all three tiers back to external stores after each turn.'''
    EPISODIC_STORE[user_id]   = memory['episodic']
    SEMANTIC_STORE[user_id]   = memory['semantic']
    PROCEDURAL_STORE[user_id] = memory['procedural']

loaded = load_all_memory('user_sam')
print('Loaded memory for user_sam:')
print(f'  episodic:   {loaded["episodic"]}')
print(f'  semantic:   {loaded["semantic"]}')
print(f'  procedural: {loaded["procedural"]}')

## Exercises

**Exercise 1** — Add a 4th memory tier called `"working"` — short-term scratch space that resets after each session (like human working memory). Add `working: list[str]` to `MemoryStore` and update the classifier to route `"temporary notes"` to it. The `load_all_memory` function should always return `working: []` since it resets each session.

**Exercise 2** — Implement a `forget` function that removes a specific fact from the semantic tier. This is the GDPR "right to be forgotten" use case: the user asks the agent to stop remembering a piece of personal information.

In [ ]:
# Exercise 2 — Forget a semantic fact
def forget_fact(memory: MemoryStore, key: str) -> MemoryStore:
    '''Remove a specific fact from semantic memory.'''
    updated_semantic = {k: v for k, v in memory['semantic'].items() if k != key}
    return {**memory, 'semantic': updated_semantic}

# Demo: add a fact, then forget it
demo_memory: MemoryStore = {
    'episodic': [],
    'semantic': {'user_name': 'Sam', 'favorite_db': 'PostgreSQL', 'location': 'Toronto'},
    'procedural': [],
}
print('Before forget:', demo_memory['semantic'])

demo_memory = forget_fact(demo_memory, 'location')
print('After forget(\'location\'):', demo_memory['semantic'])

# GDPR use case: user requests to remove their name
demo_memory = forget_fact(demo_memory, 'user_name')
print('After forget(\'user_name\'):', demo_memory['semantic'])

---

Workshop complete. You have covered examples 83–88: **Google ADK**, **Haystack**, **Agno**, **Mixture of Agents**, **Vector Memory**, and **Memory Architecture**. These examples span the M11 Framework Landscape and M7 Memory and State modules.